# nano Beatrix v1.1 — the note row: framing the text's own facts

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_v1_1_notes.ipynb)

The carry arms (2026-10-17) taught a v1.1 craft to write its own carries of a frame it was given: the far question answered whole in 0.28 to 0.59 of rows where the frames arms and the controls answer none. Every frame so far held the bed's synthetic records; the text around a frame never supplied its content. THE NOTE ROW changes what a frame holds: a fact the text itself states. The composer's extractor finds definitional facts in each row's own text (a definition, an abbreviation, a quantity, a capital, a place, the founding year of an institution; never a fact about a person), writes one 32-byte lookup frame holding the fact at the first grid point after the sentence that states it, carries restating it on the grid every 384 to 640 bytes, and asks the key in the last 512 bytes. The question's distance to the fact follows where the fact falls in the text, so one data form trains the near and the far read. A text with no usable fact stays plain.

| craft (one a body) | starts from | its phase |
|---|---|---|
| nano-<body>-v1.1-cf-nt | the carry arm's end state (step 65,237; it writes carries already) | note rows three in four where the text holds a fact, the rest plain; 1.0 billion bytes, steps 65,237 to 69,052 |
| nano-<body>-v1.1-nt | the diet's end (step 61,422) | the same phase from a body that never saw a frame written by itself (steps 61,422 to 65,237) |

The gauge reads 192 held-out documents that hold a fact in two modes, the note GIVEN (the body carries it itself) and the note ABSENT (the body must notice the fact and write it), each with the whole value by the question's distance to the fact, the rows whose own note held the key, and the per-row dump; the synthetic carry gauge and the frame gauges read beside them. C2 reads the START STATE in both modes before any training: that is the real-word read, whether a body trained on synthetic frames reads a real-word frame at all.

The pass rule, written before the run: READS REAL WORDS when the start state, the note given, answers the far bin (2,048 bytes or more) whole in 0.30 of rows or more; WRITES NOTES when at the end, the note absent, the body's own note holds the key in 0.50 of rows or more; ANSWERS NEW DATA when at the end, the note absent, the far bin is answered whole in 0.20 of rows or more; TEXT when the held-out bits stay within 0.01 of the carry arm's end; CARRY KEPT when the synthetic far question stays within 0.15 of the start. The readings: all met, the body frames new data from its context; READS REAL WORDS not met at the start but ANSWERS NEW DATA met at the end, the skill had to be taught on real words and was; WRITES NOTES met without ANSWERS NEW DATA, the notes are written but not read back, the carry dose or the phase length is next; neither met, the pattern set or the start is wrong and the extractor's yield is the first thing to read.

Length: about 27 minutes a body on the card the frames train ran on, plus the preflight reads (about ten minutes: two note reads and the carry read on the start state) and the end reads. Needs: the carry arm's end state in the repo (C2 checks), and HF_TOKEN in this notebook's secrets with write access to AbstractPhil/alephllm-nano-castor-pollux.

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.41"
import torch
import geolip.nano_beatrix as N
import geolip.alephllm as A
from geolip.nano_beatrix import nano_v1 as V1
print('geolip.nano_beatrix', N.__version__, '| geolip.alephllm', A.__version__, '| torch', torch.__version__,
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 41), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()                                   # the value is never printed
assert HF_TOKEN, 'HF_TOKEN missing: the start states are read from the repo and the continuations ship to it'
os.environ['HF_TOKEN'] = HF_TOKEN

# ============ THE DECISION BLOCK — every setting from the record, with its source beside it ============
OUT = '/content/nano_v1_1_notes'                     # the session folder: the corpus, the runs, the bed's gauge folder
REPO = V1.REPO                                        # AbstractPhil/alephllm-nano-castor-pollux: the v1.1 folders read, the new ones written
NAMES = ['nano-pollux-dual-v1.1-cf-nt']              # one body first, from its carry arm's end; all four: list(V1.CRAFTS_NOTE2); from the diet's end: list(V1.CRAFTS_NOTE)
MB, GA = 16, 4                                        # the 2s shape, the fastest on the v1.1 benchmark (262,144 bytes a step)
GAUGE_EVERY = 2000                                    # the bed's gauges at 62,000 and 64,000 (every full eval) and at the end
CORPUS_BYTES = 1.0e9                                  # the frames corpus (fineweb-edu, the held-out head skipped), built once in C2
MAX_HOURS = None                                      # this session's budget in hours (None: until every craft ends or the cell is stopped)
print('decision block:', len(NAMES), 'crafts:', ', '.join(NAMES), '| the phase', V1.CONT_TOKENS / 1e9, 'B bytes |', MB, 'x', GA,
      '| gauges every', GAUGE_EVERY, 'steps | corpus', CORPUS_BYTES / 1e9, 'GB | ships to', REPO)

In [ ]:
# C2 (P) — PREFLIGHT: every start state in the repo, the held-out head, the frames corpus (built once; about a minute or two
# a gigabyte from the stream), the rows' composition pace against the card's, the push probe. Nothing trains here.
import json, time
from huggingface_hub import HfApi
from geolip.nano_beatrix.bed_corpus import write_heldout, build as build_corpus
from geolip.nano_beatrix import frames_data as FD
api = HfApi(token=HF_TOKEN)
files = set(api.list_repo_files(REPO))
missing = []
for n in NAMES:
    spec = V1.cont_init(n, MB * GA * V1.CTX)
    path = f"{spec['prefix']}/{spec['payload']}"
    if path not in files:
        missing.append(path)
assert not missing, f'not every start state is in the repo: missing {sorted(set(missing))}'
print('start states: every source craft has its end state at step', spec['step'])
held = write_heldout(f'{OUT}/bed')
folder = f'{OUT}/frames_corpus'
if not os.path.exists(f'{folder}/bed_corpus.bin'):
    build_corpus(folder, int(CORPUS_BYTES))
packed = FD.load_corpus(folder)
n, t0 = 0, time.time()
while time.time() - t0 < 10:
    FD.row_bytes(packed, 1337, n, FD.P_FRAMED, FD.CARRY, carry=True)
    n += 1
rate = n / (time.time() - t0)
need = 879_000 / FD.ROW_BYTES                         # the fastest craft's rows a second (Castor, 879k bytes a second)
print(f'frames corpus {len(packed) / 1e9:.2f} GB; the frames rows compose at {rate:,.0f} a second against {need:.0f} needed'
      + ('' if rate > 1.5 * need else ' — TOO SLOW: the card will wait on the data'))

PUSH_OK = False
path = 'preflight/push_probe_v1_1_notes.json'        # its own probe file
body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                   'compose_rows_s': rate, 'crafts': NAMES}).encode()
api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano v1.1 notes: push probe')
PUSH_OK = path in set(api.list_repo_files(REPO))
print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)

# the extractor's yield on the training corpus (the composer's contextual system): the share of drawn rows that hold a usable fact,
# the kinds of fact, a few examples (key, value, the question's distance to the fact)
import collections
import numpy as np
kinds, have, ex = collections.Counter(), 0, []
for j in range(300):
    p = int(np.random.default_rng([3, j]).integers(0, len(packed) - 2 * FD.ROW_BYTES - 2))
    r = FD.note_row(np.random.default_rng([3, j, 1]), packed[p:p + 2 * FD.ROW_BYTES].tobytes(), FD.NOTE, FD.ROW_CTX)
    if r:
        have += 1
        kinds[r[1]['fact_kind']] += 1
        if len(ex) < 8:
            ex.append((r[1]['key'], r[1]['early_vals'][0], r[1]['fact_dist']))
print(f"note rows: {have / 300:.0%} of 300 drawn rows hold a usable fact (the rest train plain); kinds {dict(kinds)}; examples {ex}")
assert have >= 30, 'the extractor finds a fact in fewer than one row in ten: read the examples before training'

# THE REAL-WORD READ, before any training: the start state read on 192 held-out documents that hold a fact, the note GIVEN (the body
# carries it itself) and the note ABSENT (the body must notice the fact and write it), beside its synthetic carry gauge; about ten
# minutes. The far bin of the note-given read is the mark READS REAL WORDS.
src = V1.ALL[NAMES[0]]['source']
rec = V1.gauge_stored(src, spec['step'], OUT, repo=REPO, carry=True, token=HF_TOKEN, extra=dict(note_gauge=1, carry_dump=1))
KEYS = ('n', 'n_scanned', 'full_value', 'first_byte', 'value_bits', 'carries_per_row', 'rows_with_carries', 'rows_with_key', 'carry_exact', 'open_prob', 'dist_latest')
for blk in ('note_planted', 'note_self', 'carry'):
    b = rec.get(blk) or {}
    print(blk, {k: (round(v, 3) if isinstance(v, float) else v) for k, v in b.items() if k in KEYS},
          'by distance to the fact', {k: (d['n'], None if d['full_value'] is None else round(d['full_value'], 2)) for k, d in (b.get('by_dist') or {}).items()})
far = ((rec.get('note_planted') or {}).get('by_dist') or {}).get('far_ge2048') or {}
print('READS REAL WORDS at the start:', 'MET' if (far.get('full_value') or 0) >= 0.30 else 'NOT MET', f"(far bin whole {far.get('full_value')}, n {far.get('n')}; the bar 0.30)")

In [ ]:
# C3 (T) — THE CONTINUATIONS: each craft starts from its v1.1 craft's final state (its first row: the before, read by the gauges), then
# runs its phase; the trainer's lines and the bed's gauge lines ([v1]) print as they come. Rerun the cell to continue (a new session
# too). The cell's stop button saves the running craft's state before it stops.
assert PUSH_OK, 'the push probe did not read back: nothing would ship'
HIST = V1.run(OUT, HF_TOKEN, NAMES, micro_batch=MB, grad_accum=GA, repo=REPO, gauge_every=GAUGE_EVERY, max_hours=MAX_HOURS,
              frames_corpus=f'{OUT}/frames_corpus')
from IPython.display import Markdown, display
display(Markdown(V1.cont_table(HIST)))

In [ ]:
# C4 — BEFORE AND AFTER: the craft's boundary rows from the repo (the start = its source's end, the end of the note phase), the note
# reads in both modes, the synthetic carry read and the frame reads side by side, the marks; then the end state's per-row dumps pulled
# here for a look at what the body wrote. Run any time; it trains nothing.
import json
from IPython.display import Markdown, display
from huggingface_hub import snapshot_download
hub = snapshot_download(REPO, token=HF_TOKEN, local_dir=f'{OUT}/hub', allow_patterns=[f'{n}/runs/*' for n in NAMES] + [f'{n}/v1_boundaries.json' for n in NAMES])
TPS = MB * GA * V1.CTX
COLS = [('held-out bits', 'bed/text/heldout_bits', 4), ('copy', 'bed/frame/copy_whole', 2), ('1,024-byte frame', 'bed/scaling/frame_1024_whole', 2),
        ('carry: far whole', 'bed/carry/full_value', 2), ('carry: carries a row', 'bed/carry/carries_per_row', 2),
        ('note given: whole', 'bed/note_planted/full_value', 2), ('note given: far bin whole', 'bed/note_planted/far_ge2048_full_value', 2),
        ('note absent: whole', 'bed/note_self/full_value', 2), ('note absent: far bin whole', 'bed/note_self/far_ge2048_full_value', 2),
        ('note absent: near bin whole', 'bed/note_self/near_lt1024_full_value', 2), ('note absent: notes a row', 'bed/note_self/carries_per_row', 2),
        ('note absent: rows whose note held the key', 'bed/note_self/rows_with_key', 2), ('note absent: exact notes', 'bed/note_self/carry_exact', 2)]
for n in NAMES:
    p = f'{hub}/{n}/v1_boundaries.json'
    rows = json.load(open(p, encoding='utf-8')) if os.path.exists(p) else []
    start = next((e for e in rows if e.get('step') == V1.plan_steps(V1.ALL[n]['source'], TPS)), None)
    end = next((e for e in rows if e.get('phase') == V1.CONT_PHASE['nt'][0]), None)
    t = ['| read | start (the source\'s end) | end of the note phase | change |', '|---|---|---|---|']
    for label, k, nd in COLS:
        a = (start or {}).get('bed', {}) .get(k) if start else None
        b = (end or {}).get('bed', {}).get(k) if end else None
        f = lambda v: '-' if v is None else f'{v:.{nd}f}'
        d = '-' if a is None or b is None else f'{b - a:+.{nd}f}'
        t.append(f'| {label} | {f(a)} | {f(b)} | {d} |')
    display(Markdown(f'**{n}**\n\n' + '\n'.join(t)))
    if end:
        eb = end.get('bed', {})
        marks = [('WRITES NOTES', eb.get('bed/note_self/rows_with_key'), 0.50), ('ANSWERS NEW DATA', eb.get('bed/note_self/far_ge2048_full_value'), 0.20)]
        print(n, ' | '.join(f"{m}: {'-' if v is None else ('MET' if v >= bar else 'NOT MET')} ({v if v is None else round(v, 3)} against {bar})" for m, v, bar in marks))
        d = V1.gauge_stored(n, V1.plan_steps(n, TPS), OUT, repo=REPO, carry=True, token=HF_TOKEN, extra=dict(note_gauge=1, carry_dump=1))
        for blk in ('note_self', 'note_planted'):
            dump = (d.get(blk) or {}).get('dump')
            if dump:
                lines = [json.loads(l) for l in open(f"{OUT}/bed/{dump['jsonl']}", encoding='utf-8')]
                shown = [l for l in lines if l['carries']][:4]
                print(blk, 'rows with a note', sum(1 for l in lines if l['carries']), 'of', len(lines))
                for l in shown:
                    print('  key', l['key'], '| truth', l['truth'], '| answer', l['answer'], '| first note', l['carries'][0]['inserted'][:40].replace('\x00', ' '))
display(Markdown(V1.table({n: (json.load(open(f'{hub}/{n}/v1_boundaries.json', encoding='utf-8')) if os.path.exists(f'{hub}/{n}/v1_boundaries.json') else []) for n in NAMES})))
%load_ext tensorboard
%tensorboard --logdir {hub}

### Changing the run

Edit the decision block in C1, rerun C1 (the install is cached), then C2 and C3. A craft already in the repo resumes; it never restarts.

| to | the edit in C1 |
|---|---|
| all four bodies from their carry arms | `NAMES = list(V1.CRAFTS_NOTE2)` (about two hours) |
| the single-phase form, from the diet's end | `NAMES = ['nano-pollux-dual-v1.1-nt']`, or all four `list(V1.CRAFTS_NOTE)` |
| another single body | `NAMES = ['nano-castor-v1.1-cf-nt']`, or `nano-pollux-canon-v1.1-cf-nt`, or `nano-pollux-dual-canon-v1.1-cf-nt` |
| a session budget | `MAX_HOURS = 1` |

The session folder is the Colab disk: the repo holds the records.